# nvMolKit + Nemotron

This notebook demonstrates a guided chemistry agent using NVIDIA Nemotron to call a small set of allow-listed scientific tools backed by nvMolKit on an NVIDIA GPU. Nemotron first reads the BioNeMo Agent Toolkit skill for nvMolKit, learning the library's supported operations, API boundaries, and GPU requirements. It then works through a molecular library one analysis at a time: validating the sample, generating Morgan fingerprints, measuring all-pairs Tanimoto similarity, identifying structural clusters, and generating and minimizing representative conformers.

Each stage follows the same transparent pattern. The notebook defines a bounded scientific function; Nemotron requests that function through a structured tool call; the notebook validates and executes it; the result is visualized immediately; and Nemotron provides a short interpretation. A final synthesis combines the numerical results from every stage into a detailed scientific discussion.

Brev supplies the GPU environment, nvMolKit performs the batched GPU chemistry operations, RDKit handles molecule parsing and display preparation, and the notebook enforces the execution and scientific-safety boundaries. Nemotron chooses validated tool parameters and explains results, but it does not execute arbitrary Python.

This is a cheminformatics demonstration, not a benchmark or validated scientific study. Fingerprints, similarities, clusters, force-field energies, and candidate geometries are computational outputs. They do not establish binding, biological activity, ADMET properties, efficacy, safety, synthesizability, or clinical relevance.

## 1. Preflight

Run this notebook in Brev-managed Jupyter on a compatible NVIDIA GPU. The hosted NVIDIA Developer API key is read from the environment when available or entered through a hidden notebook prompt; it is never displayed or stored by the notebook.

In [ ]:
import json
import os
import sys
from getpass import getpass
from pathlib import Path

PROJECT_ROOT = None
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (
        (candidate / "data" / "sample_molecules.csv").is_file()
        and (candidate / "skills" / "nvmolkit" / "SKILL.md").is_file()
        and (candidate / "demo_agent.py").is_file()
    ):
        PROJECT_ROOT = candidate
        break
if PROJECT_ROOT is None:
    raise RuntimeError(
        "Run this notebook from the repository root or its notebooks/ directory."
    )

sys.path.insert(0, str(PROJECT_ROOT))
DATA_PATH = PROJECT_ROOT / "data" / "sample_molecules.csv"
SKILL_PATH = PROJECT_ROOT / "skills" / "nvmolkit" / "SKILL.md"

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import py3Dmol
import seaborn as sns
import torch
from IPython.display import Markdown, display
from rdkit import Chem
from rdkit.Chem import AllChem, Draw
from rdkit.Chem.rdDistGeom import ETKDGv3
from rdkit.Geometry import Point3D

from demo_agent import (
    ClusterArgs,
    FingerprintArgs,
    PrepareSampleArgs,
    ReadSkillArgs,
    SimilarityArgs,
    request_and_execute_step,
    request_brief_interpretation,
)
import nvmolkit
from nvmolkit.clustering import fused_butina
from nvmolkit.embedMolecules import EmbedMolecules
from nvmolkit.fingerprints import MorganFingerprintGenerator
from nvmolkit.mmffOptimization import MMFFOptimizeMoleculesConfs
from nvmolkit.similarity import crossTanimotoSimilarity
from nvmolkit.types import CoordinateOutput

if not torch.cuda.is_available():
    raise RuntimeError("A CUDA-capable NVIDIA GPU is required.")
if torch.version.cuda is None:
    raise RuntimeError("The installed PyTorch build does not expose CUDA.")
cuda_capability = torch.cuda.get_device_capability(0)
if cuda_capability < (7, 0):
    raise RuntimeError("nvMolKit requires NVIDIA compute capability 7.0 or newer.")
if nvmolkit.__version__ != "0.5.0":
    raise RuntimeError(
        f"This notebook is pinned to nvMolKit 0.5.0; found {nvmolkit.__version__}."
    )

print("GPU:", torch.cuda.get_device_name(0))
print("CUDA capability:", cuda_capability)
print("PyTorch/CUDA:", torch.__version__, torch.version.cuda)
print("nvMolKit:", nvmolkit.__version__)

probe_molecules = [
    Chem.MolFromSmiles(smiles) for smiles in ("CCO", "CCN", "c1ccccc1")
]
if any(molecule is None for molecule in probe_molecules):
    raise RuntimeError("The fixed GPU probe molecules did not parse.")
probe_fingerprints = MorganFingerprintGenerator(
    radius=2, fpSize=1024
).GetFingerprints(probe_molecules)
torch.cuda.synchronize()
if tuple(probe_fingerprints.torch().shape) != (3, 32):
    raise RuntimeError("The L4-compatible nvMolKit fingerprint probe failed.")
print("GPU fingerprint probe passed:", tuple(probe_fingerprints.torch().shape))

In [ ]:
api_key = os.environ.get("NVIDIA_API_KEY", "").strip()
if not api_key:
    api_key = getpass(
        "Hosted NVIDIA Developer API key from the Nemotron build.nvidia.com "
        "model page (starts with nvapi-; bare key only; input hidden): "
    ).strip()
    if not api_key:
        raise RuntimeError("A hosted NVIDIA Developer API key is required.")
model = "nvidia/nemotron-3-nano-30b-a3b"

## 2. Nemotron learns the nvMolKit skill

**Task.** Ask Nemotron to read the fixed, pinned BioNeMo Agent Toolkit skill before any sample analysis. The local executor exposes the five operations used by this notebook and the GPU-only boundary; no model-selected path is accepted.

In [ ]:
def read_nvmolkit_skill(args: ReadSkillArgs):
    del args
    skill_text = SKILL_PATH.read_text(encoding="utf-8")
    capabilities = [
        {
            "Capability": "Morgan fingerprints",
            "Entry point": "MorganFingerprintGenerator",
            "Role": "Molecular representation",
        },
        {
            "Capability": "Tanimoto similarity",
            "Entry point": "crossTanimotoSimilarity",
            "Role": "Pairwise structural similarity",
        },
        {
            "Capability": "Butina clustering",
            "Entry point": "fused_butina",
            "Role": "Structural grouping",
        },
        {
            "Capability": "ETKDG embedding",
            "Entry point": "EmbedMolecules",
            "Role": "Candidate 3D conformers",
        },
        {
            "Capability": "MMFF94 optimization",
            "Entry point": "MMFFOptimizeMoleculesConfs",
            "Role": "Force-field minimization",
        },
    ]
    summary = {
        "pinned_revision": "ce151c15470991c8cb9a0efdd531a124c346ca5b",
        "source": (
            "https://github.com/NVIDIA-BioNeMo/bionemo-agent-toolkit/blob/"
            "ce151c15470991c8cb9a0efdd531a124c346ca5b/"
            "library-skills/nvMolKit/SKILL.md"
        ),
        "gpu_boundary": (
            "An NVIDIA GPU with compute capability 7.0 or newer is required. "
            "There is no CPU fallback."
        ),
        "capabilities": capabilities,
        "figure_context": {
            "visual": "capability table",
            "rows": 5,
            "columns": ["Capability", "Entry point", "Role"],
        },
    }
    return {"skill_text": skill_text, "summary": summary}

In [ ]:
# Validation completes before the executor runs.
skill_decision, skill_artifact = request_and_execute_step(
    api_key,
    tool_name="read_nvmolkit_skill",
    task_prompt=(
        "Read the pinned nvMolKit skill and ground the guided chemistry workflow "
        "in its documented capabilities and GPU-only limitations."
    ),
    context={
        "artifact": "fixed vendored nvMolKit skill",
        "expected_capabilities": 5,
    },
    executor=read_nvmolkit_skill,
    model=model,
)
display(Markdown(f"**Requested tool:** `{skill_decision.tool_name}`"))
display(
    Markdown(
        "**Validated arguments:** `"
        + json.dumps(skill_decision.arguments.model_dump(mode="json"), sort_keys=True)
        + "`"
    )
)

In [ ]:
json.dumps(skill_artifact["summary"], allow_nan=False)
capability_table = pd.DataFrame(
    [
        {
            "Capability": "Morgan fingerprints",
            "Entry point": "MorganFingerprintGenerator",
            "Role": "Molecular representation",
        },
        {
            "Capability": "Tanimoto similarity",
            "Entry point": "crossTanimotoSimilarity",
            "Role": "Pairwise structural similarity",
        },
        {
            "Capability": "Butina clustering",
            "Entry point": "fused_butina",
            "Role": "Structural grouping",
        },
        {
            "Capability": "ETKDG embedding",
            "Entry point": "EmbedMolecules",
            "Role": "Candidate 3D conformers",
        },
        {
            "Capability": "MMFF94 optimization",
            "Entry point": "MMFFOptimizeMoleculesConfs",
            "Role": "Force-field minimization",
        },
    ]
)
display(capability_table)
skill_grounding = {
    "pinned_revision": skill_artifact["summary"]["pinned_revision"],
    "gpu_boundary": skill_artifact["summary"]["gpu_boundary"],
    "capabilities": [
        capability["Entry point"]
        for capability in skill_artifact["summary"]["capabilities"]
    ],
}

In [ ]:
try:
    skill_interpretation = request_brief_interpretation(
        api_key,
        skill_decision,
        {
            "summary": skill_artifact["summary"],
            "full_skill_text": skill_artifact["skill_text"],
        },
        {
            **skill_artifact["summary"]["figure_context"],
            "interpretation_scope": (
                "Explain the documented capabilities and GPU/API limitations."
            ),
        },
        model=model,
    )
except Exception:
    skill_interpretation = "Interpretation unavailable"
display(Markdown(skill_interpretation))

## 3. Molecular sample

**Task.** Ask Nemotron to validate and preview exactly 24 entries from the fixed bundled CSV. The executor checks the raw 256-row shape, excludes invalid SMILES while preserving metadata, and makes the exclusions visible.

In [ ]:
def prepare_molecular_sample(args: PrepareSampleArgs):
    # DATA_PATH is fixed; the model cannot choose a file or filesystem location.
    raw_sample = pd.read_csv(DATA_PATH)
    if (
        len(raw_sample) != 256
        or list(raw_sample.columns) != ["molecule_id", "smiles"]
    ):
        raise RuntimeError(
            "The bundled sample must contain exactly 256 rows and the expected columns."
        )

    parsed_molecules = [
        Chem.MolFromSmiles(str(smiles)) for smiles in raw_sample["smiles"]
    ]
    valid_mask = np.array(
        [molecule is not None for molecule in parsed_molecules], dtype=bool
    )
    # Invalid SMILES never enter GPU artifacts; their identifiers remain reportable.
    excluded_identifiers = [
        str(identifier)
        for identifier in raw_sample.loc[~valid_mask, "molecule_id"].tolist()
    ]
    molecules = [
        molecule for molecule in parsed_molecules if molecule is not None
    ]
    if not molecules:
        raise RuntimeError("The bundled sample produced zero valid molecules.")

    filtered_sample = raw_sample.loc[valid_mask].reset_index(drop=True).copy()
    summary = {
        "raw_rows": int(len(raw_sample)),
        "valid_molecules": int(len(molecules)),
        "invalid_molecules": int(len(excluded_identifiers)),
        "excluded_identifiers": excluded_identifiers,
        "preview_count": int(args.preview_count),
        "figure_context": {
            "visual": "2D molecule grid",
            "displayed_molecules": int(args.preview_count),
            "molecules_per_row": 6,
            "scope": "first valid molecules in bundled input order",
        },
    }
    return {
        "frame": filtered_sample,
        "molecules": molecules,
        "summary": summary,
    }

In [ ]:
# Validation completes before the executor runs.
sample_decision, sample_artifact = request_and_execute_step(
    api_key,
    tool_name="prepare_molecular_sample",
    task_prompt=(
        "Validate the fixed 256-row sample and preview exactly 24 valid molecules. "
        "Report all invalid SMILES exclusions."
    ),
    context={
        "skill_grounding": skill_grounding,
        "fixed_dataset": "data/sample_molecules.csv",
        "raw_rows_expected": 256,
        "preview_count": 24,
    },
    executor=prepare_molecular_sample,
    model=model,
)
display(Markdown(f"**Requested tool:** `{sample_decision.tool_name}`"))
display(
    Markdown(
        "**Validated arguments:** `"
        + json.dumps(sample_decision.arguments.model_dump(mode="json"), sort_keys=True)
        + "`"
    )
)

In [ ]:
json.dumps(sample_artifact["summary"], allow_nan=False)
excluded_text = (
    ", ".join(sample_artifact["summary"]["excluded_identifiers"])
    if sample_artifact["summary"]["excluded_identifiers"]
    else "none"
)
display(
    Markdown(
        f"**Invalid SMILES:** {sample_artifact['summary']['invalid_molecules']}; "
        f"**excluded identifiers:** {excluded_text}"
    )
)
display(
    pd.Series(
        {
            "Raw rows": sample_artifact["summary"]["raw_rows"],
            "Valid molecules": sample_artifact["summary"]["valid_molecules"],
            "Invalid molecules": sample_artifact["summary"]["invalid_molecules"],
            "Previewed molecules": sample_artifact["summary"]["preview_count"],
        },
        name="Sample",
    ).to_frame()
)
display(
    Draw.MolsToGridImage(
        sample_artifact["molecules"][:24],
        legends=sample_artifact["frame"]["molecule_id"].iloc[:24].tolist(),
        molsPerRow=6,
        subImgSize=(220, 180),
    )
)

In [ ]:
try:
    sample_interpretation = request_brief_interpretation(
        api_key,
        sample_decision,
        sample_artifact["summary"],
        {
            **sample_artifact["summary"]["figure_context"],
            "interpretation_scope": (
                "The 24-molecule preview cannot establish whole-library chemistry."
            ),
        },
        model=model,
    )
except Exception:
    sample_interpretation = "Interpretation unavailable"
display(Markdown(sample_interpretation))

## 4. Mapping molecular similarity

The following three guided calls build one representation, measure all-pairs similarity, and identify clusters. Each executor consumes only validated arguments plus local artifacts from earlier stages.

### 4.1 Morgan fingerprints

**Task.** Ask Nemotron to choose strict fingerprint parameters, recommending radius 2 and 1,024 bits. The executor computes packed Morgan fingerprints for every valid molecule and summarizes representation density without inferring biological activity.

In [ ]:
def compute_morgan_fingerprints(args: FingerprintArgs):
    generator = MorganFingerprintGenerator(
        radius=args.fingerprint_radius,
        fpSize=args.fingerprint_size,
    )
    fingerprints = generator.GetFingerprints(sample_artifact["molecules"])
    fingerprint_tensor = fingerprints.torch()
    expected_shape = (
        len(sample_artifact["molecules"]),
        args.fingerprint_size // 32,
    )
    if tuple(fingerprint_tensor.shape) != expected_shape:
        raise RuntimeError("The packed Morgan fingerprint tensor shape was unexpected.")

    # Each GPU-resident int32 word packs 32 hashed fingerprint bits.
    packed_words = fingerprint_tensor.to(torch.int64) & 0xFFFFFFFF
    bit_positions = torch.arange(
        32, dtype=torch.int64, device=fingerprint_tensor.device
    )
    active_bits_gpu = (
        (packed_words.unsqueeze(-1) >> bit_positions) & 1
    ).sum(dim=(1, 2))
    # Synchronize GPU work before moving active hashed bits into a host summary.
    torch.cuda.synchronize()
    active_bits = active_bits_gpu.cpu().numpy().astype(np.int64)

    summary = {
        "tensor_shape": [int(value) for value in fingerprint_tensor.shape],
        "radius": int(args.fingerprint_radius),
        "size": int(args.fingerprint_size),
        "molecule_count": int(len(sample_artifact["molecules"])),
        "device": str(fingerprint_tensor.device),
        "min_active_bits": int(active_bits.min()),
        "median_active_bits": float(np.median(active_bits)),
        "mean_active_bits": float(np.mean(active_bits)),
        "max_active_bits": int(active_bits.max()),
        "figure_context": {
            "visual": "histogram",
            "x_axis": "active hashed bits per molecule",
            "y_axis": "molecule count",
            "bins": 20,
        },
    }
    return {
        "fingerprints": fingerprints,
        "tensor": fingerprint_tensor,
        "active_bits": active_bits,
        "summary": summary,
    }

In [ ]:
# Validation completes before the executor runs.
fingerprint_decision, fingerprint_artifact = request_and_execute_step(
    api_key,
    tool_name="compute_morgan_fingerprints",
    task_prompt=(
        "Generate Morgan fingerprints for all valid molecules. Use the recommended "
        "radius 2 and 1024-bit representation unless a different allowed value is "
        "scientifically justified by the bounded context."
    ),
    context={
        "skill_grounding": skill_grounding,
        "sample_summary": sample_artifact["summary"],
        "recommended": {"fingerprint_radius": 2, "fingerprint_size": 1024},
    },
    executor=compute_morgan_fingerprints,
    model=model,
)
display(Markdown(f"**Requested tool:** `{fingerprint_decision.tool_name}`"))
display(
    Markdown(
        "**Validated arguments:** `"
        + json.dumps(
            fingerprint_decision.arguments.model_dump(mode="json"), sort_keys=True
        )
        + "`"
    )
)

In [ ]:
json.dumps(fingerprint_artifact["summary"], allow_nan=False)
display(
    pd.Series(
        {
            "Tensor shape": fingerprint_artifact["summary"]["tensor_shape"],
            "Radius": fingerprint_artifact["summary"]["radius"],
            "Fingerprint bits": fingerprint_artifact["summary"]["size"],
            "Minimum active bits": fingerprint_artifact["summary"]["min_active_bits"],
            "Median active bits": fingerprint_artifact["summary"]["median_active_bits"],
            "Mean active bits": fingerprint_artifact["summary"]["mean_active_bits"],
            "Maximum active bits": fingerprint_artifact["summary"]["max_active_bits"],
        },
        name="Morgan fingerprints",
    ).to_frame()
)
plt.figure(figsize=(7, 4))
plt.hist(
    fingerprint_artifact["active_bits"],
    bins=20,
    color="#76B900",
    edgecolor="black",
)
plt.title("Active Morgan fingerprint bits per molecule")
plt.xlabel("Active hashed bits")
plt.ylabel("Molecule count")
plt.tight_layout()
plt.show()

In [ ]:
try:
    fingerprint_interpretation = request_brief_interpretation(
        api_key,
        fingerprint_decision,
        fingerprint_artifact["summary"],
        {
            **fingerprint_artifact["summary"]["figure_context"],
            "interpretation_scope": (
                "Interpret representation and density only; do not infer biological activity."
            ),
        },
        model=model,
    )
except Exception:
    fingerprint_interpretation = "Interpretation unavailable"
display(Markdown(fingerprint_interpretation))

### 4.2 All-pairs Tanimoto similarity

**Task.** Ask Nemotron to request the argument-free similarity stage. The executor reuses the local GPU fingerprint artifact, validates the square 0–1 matrix, excludes trivial diagonal self-similarity from statistics, and reports the most similar off-diagonal molecule pair.

In [ ]:
def compute_tanimoto_similarity(args: SimilarityArgs):
    del args
    similarity_result = crossTanimotoSimilarity(
        fingerprint_artifact["fingerprints"]
    )
    torch.cuda.synchronize()
    similarity_matrix = similarity_result.torch().cpu().numpy()
    molecule_count = len(sample_artifact["molecules"])

    if similarity_matrix.shape != (molecule_count, molecule_count):
        raise RuntimeError("The all-pairs Tanimoto matrix shape was unexpected.")
    if not np.isfinite(similarity_matrix).all():
        raise RuntimeError("The Tanimoto matrix contains non-finite values.")
    if not np.allclose(
        similarity_matrix, similarity_matrix.T, rtol=0, atol=1e-7
    ):
        raise RuntimeError("The Tanimoto matrix is not symmetric.")
    if np.any((similarity_matrix < 0) | (similarity_matrix > 1)):
        raise RuntimeError("Tanimoto values must remain on the 0-1 scale.")

    # Self-similarity on the diagonal is trivially 1.0, so exclude it from statistics.
    upper_rows, upper_columns = np.triu_indices(molecule_count, k=1)
    off_diagonal = similarity_matrix[upper_rows, upper_columns]
    pair_position = int(np.argmax(off_diagonal))
    first_index = int(upper_rows[pair_position])
    second_index = int(upper_columns[pair_position])
    molecule_ids = sample_artifact["frame"]["molecule_id"].astype(str).tolist()

    summary = {
        "shape": [int(value) for value in similarity_matrix.shape],
        "q1": float(np.quantile(off_diagonal, 0.25)),
        "median": float(np.median(off_diagonal)),
        "q3": float(np.quantile(off_diagonal, 0.75)),
        "p90": float(np.quantile(off_diagonal, 0.90)),
        "max_off_diagonal": float(off_diagonal[pair_position]),
        "most_similar_nonidentical_pair_ids": [
            molecule_ids[first_index],
            molecule_ids[second_index],
        ],
        "figure_context": {
            "visual": "unordered heatmap",
            "x_axis": "molecules in validated input order",
            "y_axis": "molecules in validated input order",
            "scale": [0.0, 1.0],
            "color": "Tanimoto similarity",
        },
    }
    return {
        "result": similarity_result,
        "matrix": similarity_matrix,
        "summary": summary,
    }

In [ ]:
# Validation completes before the executor runs.
similarity_decision, similarity_artifact = request_and_execute_step(
    api_key,
    tool_name="compute_tanimoto_similarity",
    task_prompt=(
        "Measure all-pairs Tanimoto similarity from the existing local fingerprint "
        "artifact without selecting any additional parameters."
    ),
    context={
        "skill_grounding": skill_grounding,
        "fingerprint_summary": fingerprint_artifact["summary"],
    },
    executor=compute_tanimoto_similarity,
    model=model,
)
display(Markdown(f"**Requested tool:** `{similarity_decision.tool_name}`"))
display(
    Markdown(
        "**Validated arguments:** `"
        + json.dumps(similarity_decision.arguments.model_dump(mode="json"), sort_keys=True)
        + "`"
    )
)

In [ ]:
json.dumps(similarity_artifact["summary"], allow_nan=False)
display(
    pd.Series(
        {
            "Matrix shape": similarity_artifact["summary"]["shape"],
            "Q1 (off-diagonal)": similarity_artifact["summary"]["q1"],
            "Median (off-diagonal)": similarity_artifact["summary"]["median"],
            "Q3 (off-diagonal)": similarity_artifact["summary"]["q3"],
            "90th percentile": similarity_artifact["summary"]["p90"],
            "Maximum off-diagonal": similarity_artifact["summary"]["max_off_diagonal"],
            "Most similar nonidentical pair": " / ".join(
                similarity_artifact["summary"][
                    "most_similar_nonidentical_pair_ids"
                ]
            ),
        },
        name="Tanimoto similarity",
    ).to_frame()
)
plt.figure(figsize=(8, 7))
sns.heatmap(
    similarity_artifact["matrix"],
    cmap="viridis",
    vmin=0,
    vmax=1,
    cbar_kws={"label": "Tanimoto similarity"},
)
plt.title("Unordered all-pairs Tanimoto similarity (validated input order)")
plt.xlabel("Molecule index in input order")
plt.ylabel("Molecule index in input order")
plt.tight_layout()
plt.show()

In [ ]:
try:
    similarity_interpretation = request_brief_interpretation(
        api_key,
        similarity_decision,
        similarity_artifact["summary"],
        {
            **similarity_artifact["summary"]["figure_context"],
            "interpretation_scope": (
                "Explain the off-diagonal distribution and most-similar pair without "
                "inferring shared biological activity."
            ),
        },
        model=model,
    )
except Exception:
    similarity_interpretation = "Interpretation unavailable"
display(Markdown(similarity_interpretation))

### 4.3 Fused Butina clusters

**Task.** Ask Nemotron to cluster the existing fingerprints, recommending a 0.50 Tanimoto-distance cutoff within the strict 0.40–0.60 range. The executor validates complete, unique assignment and reports singletons explicitly because cluster fragmentation depends on the selected cutoff.

In [ ]:
def cluster_with_fused_butina(args: ClusterArgs):
    fingerprints = fingerprint_artifact["fingerprints"]
    cutoff = float(args.cluster_cutoff)
    # The cutoff is a Tanimoto-distance threshold: similarity > 1 - cutoff.
    # Lowering it requires greater similarity and can create more singletons.
    clusters, reported_cluster_sizes = fused_butina(
        fingerprints.torch(), cutoff=cutoff
    )
    torch.cuda.synchronize()

    molecule_count = len(sample_artifact["molecules"])
    assigned_indices = [
        int(molecule_index)
        for cluster in clusters
        for molecule_index in cluster
    ]
    if (
        len(assigned_indices) != molecule_count
        or sorted(assigned_indices) != list(range(molecule_count))
    ):
        raise RuntimeError("Every molecule must be assigned exactly once.")

    assignments = np.full(molecule_count, -1, dtype=int)
    for cluster_id, cluster in enumerate(clusters):
        for molecule_index in cluster:
            assignments[int(molecule_index)] = int(cluster_id)
    cluster_sizes = [int(len(cluster)) for cluster in clusters]
    singleton_count = int(sum(size == 1 for size in cluster_sizes))
    largest_cluster_sizes = sorted(cluster_sizes, reverse=True)[:15]

    summary = {
        "cutoff": cutoff,
        "cluster_count": int(len(clusters)),
        "singleton_count": singleton_count,
        "singleton_fraction": float(singleton_count / molecule_count),
        "largest_cluster_sizes": largest_cluster_sizes,
        "molecule_count": int(molecule_count),
        "figure_context": {
            "visual": "bar chart of 15 largest clusters",
            "x_axis": "cluster rank by descending size",
            "y_axis": "molecule count",
            "singleton_count": singleton_count,
            "cutoff": cutoff,
        },
    }
    return {
        "assignments": assignments,
        "clusters": clusters,
        "reported_cluster_sizes": reported_cluster_sizes,
        "summary": summary,
    }

In [ ]:
# Validation completes before the executor runs.
cluster_decision, cluster_artifact = request_and_execute_step(
    api_key,
    tool_name="cluster_with_fused_butina",
    task_prompt=(
        "Cluster the existing Morgan fingerprints with fused Butina. Use the "
        "recommended Tanimoto-distance cutoff 0.50 unless another allowed value is justified."
    ),
    context={
        "skill_grounding": skill_grounding,
        "fingerprint_summary": fingerprint_artifact["summary"],
        "similarity_summary": similarity_artifact["summary"],
        "recommended": {"cluster_cutoff": 0.50},
    },
    executor=cluster_with_fused_butina,
    model=model,
)
display(Markdown(f"**Requested tool:** `{cluster_decision.tool_name}`"))
display(
    Markdown(
        "**Validated arguments:** `"
        + json.dumps(cluster_decision.arguments.model_dump(mode="json"), sort_keys=True)
        + "`"
    )
)

In [ ]:
json.dumps(cluster_artifact["summary"], allow_nan=False)
display(
    pd.Series(
        {
            "Cutoff": cluster_artifact["summary"]["cutoff"],
            "Clusters": cluster_artifact["summary"]["cluster_count"],
            "Singletons": cluster_artifact["summary"]["singleton_count"],
            "Singleton fraction": cluster_artifact["summary"]["singleton_fraction"],
            "Molecules assigned": cluster_artifact["summary"]["molecule_count"],
        },
        name="Fused Butina clustering",
    ).to_frame()
)
top_cluster_sizes = sorted(
    [len(cluster) for cluster in cluster_artifact["clusters"]], reverse=True
)[:15]
cluster_ranks = np.arange(1, len(top_cluster_sizes) + 1)
plt.figure(figsize=(8, 4))
plt.bar(cluster_ranks, top_cluster_sizes, color="#76B900")
plt.title(
    "15 largest fused Butina clusters "
    f"(singletons: {cluster_artifact['summary']['singleton_count']})"
)
plt.xlabel("Cluster rank by descending size")
plt.ylabel("Molecule count")
plt.xticks(cluster_ranks)
plt.tight_layout()
plt.show()

In [ ]:
try:
    cluster_interpretation = request_brief_interpretation(
        api_key,
        cluster_decision,
        cluster_artifact["summary"],
        {
            **cluster_artifact["summary"]["figure_context"],
            "interpretation_scope": (
                "Discuss fragmentation, diversity, singletons, and cutoff sensitivity "
                "without biological claims."
            ),
        },
        model=model,
    )
except Exception:
    cluster_interpretation = "Interpretation unavailable"
display(Markdown(cluster_interpretation))

## 5. Conformers and MMFF94

The next guided stage will select cluster representatives, generate candidate ETKDG conformers, and minimize them with MMFF94. Those geometries and energies will remain sampled computational force-field outputs, not experimentally validated conformations or evidence of biological activity.

## 6. What the results mean

After the conformer stage is implemented, Nemotron will synthesize the JSON-safe summaries from every completed stage. The synthesis will separate molecular descriptors, similarity structure, clustering, sampled geometries, and force-field minima from binding, activity, ADMET, efficacy, safety, synthesizability, clinical, or experimental claims.